In [1]:
import sys; sys.path.insert(0, ".")
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
from constants import (
    result_path,
    ret_path,
)
from decomposition import (
    build_conditional_stats,
)
from spanning import (
    prepare_insample_spanning_df,
    run_span_bab_models,
    run_span_bab_vol_timed_models,
)
from analysis_utils import (
    stargazer_with_tstats,
)


In [2]:
# Load baseline data and cached monthly return panel
ff6_df = pd.read_csv(Path(ret_path) / "ff6_monthly_returns.csv")
res_dir = Path(result_path)
port_ret_all = pd.read_csv(res_dir / "Rg_port_ret_all.csv")

# Rg_port_ret_all.csv already carries the MKTg and LNF rows (built in build_rg_factors).
ret_ext = port_ret_all

# Prepare monthly spanning regression dataset for BAB analysis
g_spec = "Rg"
reg_df = prepare_insample_spanning_df(
    ret_ext=ret_ext,
    ff6_df=ff6_df,
    g_port=g_spec,
    ret_col="ret_ex",
)
print(f"Results below using global factor specified as {g_spec}")

Results below using global factor specified as Rg


In [3]:
# Run BAB spanning regressions (full sample, 6 models)
bab_models = run_span_bab_models(reg_df, pair_cols=("g_rf", "lnf"))
display(stargazer_with_tstats(
    models=bab_models,
    covariate_order=["Intercept", "MKT_RF", "g_rf", "lnf", "SMB", "HML", "RMW", "CMA", "MOM"],
    latex_filename="spanning_bab.tex",
))


In [4]:
# Vol-timed BAB spanning: call and show bab_vol_models1 (low/high vol split)
bab_vol_models1, bab_vol_models2, bab_vol_models3, bab_group_ym = run_span_bab_vol_timed_models(
    reg_df,
    pair_cols=("g_rf", "lnf"),
)
display(stargazer_with_tstats(
    models=bab_vol_models1,
    covariate_order=["Intercept", "MKT_RF", "g_rf", "lnf", "SMB", "HML", "RMW", "CMA", "MOM"],
    latex_filename="spanning_bab_vol_timed.tex",
))

run_span_bab_vol_timed_models: grouping by lagged BAB vol (t-1 month annualized realized vol from daily BAB), median=7.06%, low_n=120, high_n=120


In [5]:
# bab_vol_models2: full sample with dummy (below median = 1) and interactions
display(stargazer_with_tstats(
    models=bab_vol_models2,
    covariate_order=["Intercept", "low_bab_vol", "MKT_RF", "low_bab_vol:MKT_RF", "g_rf", "lnf", "low_bab_vol:g_rf", "low_bab_vol:lnf"],
    latex_filename="spanning_bab_lag_vol_dummy.tex",
))

In [6]:
# bab_vol_models3: full sample with continuous log(lag_bab_vol) and interactions
display(stargazer_with_tstats(
    models=bab_vol_models3,
    covariate_order=["Intercept", "lag_bab_vol", "MKT_RF", "lag_bab_vol:MKT_RF", "g_rf", "lnf", "lag_bab_vol:g_rf", "lag_bab_vol:lnf"],
    latex_filename="spanning_bab_lag_vol_interact.tex",
))

In [7]:
# In-sample decomposition: conditional on high vs low lagged BAB volatility (using bab_group_ym from previous cell)
stats_babvol_df = build_conditional_stats(
    ret_ext=ret_ext,
    group_ym=bab_group_ym,
    save_filename="babvol_conditional_stats_is.csv",
)



Conditional Stats (Full Excess Return, annualized)
  Months partitioned into {Low bab_vol, High bab_vol}. Each row reports that portfolio's annualized stats within the month subsample.
Portfolio        | Unconditional                 | Low bab_vol                   | High bab_vol                  | SR diff (Low bab_vol-High bab_vol)
                 |   mean(%)    vol(%)        SR |   mean(%)    vol(%)        SR |   mean(%)    vol(%)        SR |           diff    t-stat
-------------------------------------------------------------------------------------------------------------------------------------------
vwMKT-RF         |      9.63     15.68      0.61 |     10.82     11.39      0.95 |      8.43     19.08      0.44 |           0.51      1.09
LowBeta-RF       |      8.51     11.11      0.77 |     10.46      8.65      1.21 |      6.55     13.13      0.50 |     0.71$^{*}$      1.79
HighBeta-RF      |     12.03     30.40      0.40 |      9.79     23.84      0.41 |     14.27     35.88  